# Hebrew Song Decade Classification - Full Experiment Pipeline

**Goal**: Classify Hebrew Israeli songs into decades (1980s, 2000s, 2020s) using linguistic features.

**Data**: 597 songs x 137 features (basic stylistic + Dicta morphological)

**Source**: https://github.com/shon2k01/NLP

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import StratifiedKFold, cross_validate, train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.naive_bayes import MultinomialNB
from sklearn.dummy import DummyClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import classification_report, confusion_matrix, make_scorer, f1_score
from scipy.sparse import hstack, csr_matrix
import warnings
warnings.filterwarnings('ignore')

# Load data (adjust paths as needed)
ft_dicta = pd.read_csv('outputs/song_feature_table_with_dicta.csv')
corpus = pd.read_csv('israeli_songs_corpus.csv')
df = ft_dicta.merge(corpus[['song_name', 'artist_name', 'decade', 'lyrics']], 
                    on=['song_name', 'artist_name', 'decade'], how='left')
df = df.drop(columns=['tense_present_count', 'tense_present_ratio'], errors='ignore')
print(f'Dataset: {df.shape[0]} songs x {df.shape[1]} columns')
print(f'Decades: {df["decade"].value_counts().to_dict()}')

In [ ]:
# Define feature groups
FEATURE_GROUPS = {
    'basic_surface': ['word_count','unique_word_count','char_count','line_count','avg_words_per_line','median_words_per_line','max_words_per_line','min_words_per_line','avg_word_length','lexical_diversity'],
    'repetition': ['repetition_ratio','repeated_lines_count','repeated_lines_ratio','unique_lines_count','most_repeated_line_count','chorus_repetition_score','repeated_words_count','repeated_words_ratio','most_common_word_count','most_common_word_ratio'],
    'pronoun_person': ['first_person_singular_count','first_person_singular_ratio','first_person_plural_count','first_person_plural_ratio','second_person_count','second_person_ratio','third_person_count','third_person_ratio','we_vs_i_ratio','direct_address_score'],
    'semantic_lexicon': ['love_words_ratio','army_words_ratio','nature_words_ratio','religion_words_ratio','party_words_ratio','sadness_words_ratio','nostalgia_words_ratio','family_words_ratio','place_words_ratio','time_words_ratio','modern_slang_words_ratio','old_israeli_words_ratio'],
    'dicta_pos': ['pos_noun_ratio','pos_verb_ratio','pos_adj_ratio','pos_adv_ratio','pos_pron_ratio','pos_propn_ratio','pos_adp_ratio','pos_det_ratio','pos_cconj_ratio','pos_sconj_ratio','pos_aux_ratio','pos_num_ratio','pos_intj_ratio'],
    'dicta_morphology': ['tense_past_ratio','tense_future_ratio','verb_inf_ratio','verb_participle_ratio','imperative_ratio','masculine_ratio','feminine_ratio','singular_ratio','plural_ratio','first_person_morph_ratio','second_person_morph_ratio','third_person_morph_ratio'],
    'dicta_binyan': ['binyan_paal_ratio','binyan_piel_ratio','binyan_hifil_ratio','binyan_hitpael_ratio','binyan_nifal_ratio','binyan_pual_ratio','binyan_hufal_ratio'],
    'dicta_lemma_root': ['lemma_diversity','root_diversity','dicta_unknown_token_ratio'],
    'punctuation_formatting': ['punctuation_ratio','question_mark_count','exclamation_mark_count','comma_count','quote_count','parenthesis_count','english_char_ratio','digit_count'],
    'last_word': ['last_word_unique_count','last_word_repetition_ratio','avg_last_word_length'],
}
all_numeric_cols = [c for c in df.columns if c not in ['song_name','artist_name','decade','lyrics'] and df[c].dtype in ['int64','float64']]
FEATURE_GROUPS['all_numeric'] = all_numeric_cols
redundant = ['repetition_ratio','dicta_x_pos_count','dicta_x_pos_ratio','dicta_token_count','dicta_unknown_token_count','char_count','chorus_repetition_score','unique_lemma_count']
FEATURE_GROUPS['reduced_non_redundant'] = [c for c in all_numeric_cols if c not in redundant]
y = df['decade']
print(f'Feature groups: {len(FEATURE_GROUPS)}')
for name, cols in FEATURE_GROUPS.items():
    print(f'  {name}: {len(cols)} features')

In [ ]:
# Run experiments
MODELS = {
    'Majority Class': DummyClassifier(strategy='most_frequent'),
    'Stratified Random': DummyClassifier(strategy='stratified'),
    'Logistic Regression': Pipeline([('scaler', StandardScaler()), ('clf', LogisticRegression(max_iter=2000, random_state=42))]),
    'Linear SVM': Pipeline([('scaler', StandardScaler()), ('clf', LinearSVC(max_iter=5000, random_state=42))]),
    'Random Forest': RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1),
    'Gradient Boosting': GradientBoostingClassifier(n_estimators=200, random_state=42),
}
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scoring = {'accuracy': 'accuracy', 'macro_f1': make_scorer(f1_score, average='macro')}
results = []
for group_name, feature_cols in FEATURE_GROUPS.items():
    valid_cols = [c for c in feature_cols if c in df.columns]
    if not valid_cols: continue
    X = df[valid_cols].fillna(0).values
    for model_name, model in MODELS.items():
        scores = cross_validate(model, X, y, cv=cv, scoring=scoring, n_jobs=-1)
        results.append({'Feature Group': group_name, 'Model': model_name, 'Accuracy': scores['test_accuracy'].mean(), 'Macro F1': scores['test_macro_f1'].mean(), 'N Features': len(valid_cols)})
    print(f'Done: {group_name}')

lyrics = df['lyrics'].fillna('')
for tname, ngram in [('tfidf_unigrams',(1,1)),('tfidf_bigrams',(1,2))]:
    tfidf = TfidfVectorizer(max_features=5000, ngram_range=ngram, sublinear_tf=True)
    Xt = tfidf.fit_transform(lyrics)
    for mname, model in [('Logistic Regression', LogisticRegression(max_iter=2000, random_state=42)), ('Linear SVM', LinearSVC(max_iter=5000, random_state=42)), ('Multinomial NB', MultinomialNB())]:
        scores = cross_validate(model, Xt, y, cv=cv, scoring=scoring, n_jobs=-1)
        results.append({'Feature Group': tname, 'Model': mname, 'Accuracy': scores['test_accuracy'].mean(), 'Macro F1': scores['test_macro_f1'].mean(), 'N Features': Xt.shape[1]})
    print(f'Done: {tname}')

results_df = pd.DataFrame(results)
print(f'\nTotal experiments: {len(results_df)}')
print(results_df.sort_values('Accuracy', ascending=False).head(10).to_string(index=False))

In [ ]:
# Best model evaluation
X_train_idx, X_test_idx = train_test_split(range(len(df)), test_size=0.2, random_state=42, stratify=y)
y_train, y_test = y.iloc[X_train_idx], y.iloc[X_test_idx]
tfidf = TfidfVectorizer(max_features=5000, ngram_range=(1,2), sublinear_tf=True)
X_all = tfidf.fit_transform(lyrics)
model = LinearSVC(max_iter=5000, random_state=42)
model.fit(X_all[X_train_idx], y_train)
y_pred = model.predict(X_all[X_test_idx])
print('BEST MODEL: Linear SVM + TF-IDF bigrams')
print('='*50)
print(classification_report(y_test, y_pred, digits=3))
print('Confusion Matrix:')
print(pd.DataFrame(confusion_matrix(y_test, y_pred, labels=['1980s','2000s','2020s']),
                   index=['True 1980s','True 2000s','True 2020s'],
                   columns=['Pred 1980s','Pred 2000s','Pred 2020s']))

In [ ]:
# Feature importance (Random Forest)
valid_cols = [c for c in FEATURE_GROUPS['all_numeric'] if c in df.columns]
rf = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1)
rf.fit(df[valid_cols].fillna(0).values, y)
importances = pd.DataFrame({'Feature': valid_cols, 'Importance': rf.feature_importances_}).sort_values('Importance', ascending=False)
print('TOP 20 FEATURES (Random Forest Importance):')
print(importances.head(20).to_string(index=False))